# 📂 01. Data Loading and Merging
<br>
# 📓 

## 📂 Objective: Load raw CSV files (Users, Telemetry, Death Events), remove PII, map death events to telemetry windows.


### 📥 Input Data:
- `../../data/telemetry_phase_2.users.csv`- `../../data/telemetry_phase_2.telemetries.csv`- `../../data/telemetry_phase_2.deathevents.csv`
<br>
### 📤 Output Artifacts:
- `../../data/processed/merged_telemetry.csv`- `../../data/processed/merged_deathevents.csv`


In [1]:
import pandas as pd
import numpy as np
import os

print("Libraries imported")

Libraries imported


In [2]:
# File Paths
USERS_PATH = '../../data/telemetry_phase_2.users.csv'
TELEMETRY_PATH = '../../data/telemetry_phase_2.telemetries.csv'
DEATHEVENTS_PATH = '../../data/telemetry_phase_2.deathevents.csv'

PROCESSED_DIR = '../../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_TELEMETRY = os.path.join(PROCESSED_DIR, 'merged_telemetry.csv')
OUTPUT_DEATHEVENTS = os.path.join(PROCESSED_DIR, 'merged_deathevents.csv')

print(f"Processing data from: {os.path.abspath(PROCESSED_DIR)}")

Processing data from: f:\Campus\FYP\Implementation\CollectGame.Model\data\processed


### 💡 Design Decision: 30-Second Telemetry Windows

> **Context:** We define a "telemetry window" as a 30-second slice of gameplay.
>
> **Why 30 seconds?**
> * **Responsiveness:** Short enough to capture quick shifts in player attention (e.g., stopping combat to loot).
> * **Statistical Significance:** Long enough to accumulate meaningful counts (kills, distance) rather than noise.
> * **System Limit:** A balanced trade-off between granularity and storage/processing overhead.


In [3]:
# Load Data
print("Loading data...")
df_users = pd.read_csv(USERS_PATH)
df_telemetry = pd.read_csv(TELEMETRY_PATH)
df_deaths = pd.read_csv(DEATHEVENTS_PATH)

print(f"Users: {df_users.shape}")
print(f"Telemetry: {df_telemetry.shape}")
print(f"Deaths: {df_deaths.shape}")

Loading data...
Users: (65, 8)
Telemetry: (4302, 30)
Deaths: (285, 7)


In [4]:
# Remove PII
pii_cols = ['firstName', 'lastName', 'email', 'password', 'username']
cols_to_drop = [c for c in pii_cols if c in df_users.columns]
df_users_clean = df_users.drop(columns=cols_to_drop)
print(f"Removed PII columns: {cols_to_drop}")

Removed PII columns: ['firstName', 'lastName', 'email']


In [5]:
# Filter to valid users
target_user_ids = set(df_users_clean['_id'].unique())
df_telemetry = df_telemetry[df_telemetry['userId'].isin(target_user_ids)].copy()
df_deaths = df_deaths[df_deaths['userId'].isin(target_user_ids)].copy()
print(f"Filtered to valid users: {len(target_user_ids)}")

Filtered to valid users: 65


In [6]:
# Map deaths to telemetry windows
df_telemetry['timestamp'] = pd.to_numeric(df_telemetry['timestamp'], errors='coerce')
df_deaths['timestamp'] = pd.to_numeric(df_deaths['timestamp'], errors='coerce')
df_telemetry['death_count'] = 0

df_telemetry.sort_values(by=['userId', 'timestamp'], inplace=True)
df_deaths.sort_values(by=['userId', 'timestamp'], inplace=True)

count_mapped = 0
telemetry_grouped = df_telemetry.groupby('userId')

for user_id, user_deaths in df_deaths.groupby('userId'):
    if user_id in telemetry_grouped.groups:
        user_tele_indices = telemetry_grouped.groups[user_id]
        user_tele_times = df_telemetry.loc[user_tele_indices, 'timestamp'].values
        
        for death_time in user_deaths['timestamp']:
            if pd.isna(death_time): continue
            # Window: Death <= Tele < Death + 30
            mask = (user_tele_times >= death_time) & (user_tele_times < death_time + 30)
            matched = np.where(mask)[0]
            if len(matched) > 0:
                match_idx = user_tele_indices[matched[0]]
                df_telemetry.at[match_idx, 'death_count'] += 1
                count_mapped += 1

print(f"Mapped {count_mapped} death events to telemetry windows")

Mapped 0 death events to telemetry windows


In [7]:
# Merge & Save
df_merged = pd.merge(df_telemetry, df_users_clean, left_on='userId', right_on='_id', how='left')
df_deaths_merged = pd.merge(df_deaths, df_users_clean, left_on='userId', right_on='_id', how='left')

df_merged.to_csv(OUTPUT_TELEMETRY, index=False)
df_deaths_merged.to_csv(OUTPUT_DEATHEVENTS, index=False)

print(f"Saved merged files to {PROCESSED_DIR}")

Saved merged files to ../../data/processed
